In [70]:
import pandas as pd
import awswrangler as awr
import openpyxl
import shutil
import datetime as dt
import pyautogui
import time

In [71]:
def format_type(df):
    for col in df.select_dtypes(include=['string']).columns:
        df[col] = df[col].str.upper()
    return df

In [72]:
# EXTRAINDO DADOS DE ATIVOS

query_ativos = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_ATIVOS.sql"
with open(query_ativos, 'r', encoding='utf-8') as file:
    sql_ativos = file.read()   

df_ativos =awr.athena.read_sql_query(
    sql=sql_ativos,database='silver'
)

In [73]:
df_ativos = df_ativos.drop(columns=['rn'])

In [74]:
df_ativos = (
    df_ativos
    .sort_values(by=['chassi', 'inicio_vig', 'data_ativacao'], ascending=[True, False, False])
    .drop_duplicates(subset=['chassi'], keep='first')
)

In [75]:
# EXTRAINDO DADOS DE CANCELAMENTOS INTEGRAIS (CONJUNTO)

query_cancelados_integrais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_INTEGRAIS.sql"
with open(query_cancelados_integrais, 'r', encoding='utf-8') as file:
    sql_cancelados_integrais = file.read()   

df_cancelamentos_integrais =awr.athena.read_sql_query(
    sql=sql_cancelados_integrais, database='silver'
)

In [76]:
# EXTRAINDO DADOS DE CANCELAMENTOS PARCIAIS (CONJUNTO)

query_cancelados_parciais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_PARCIAIS.sql"
with open(query_cancelados_parciais, 'r', encoding='utf-8') as file:
    sql_cancelados_parciais = file.read()   

df_cancelamentos_parciais =awr.athena.read_sql_query(
    sql=sql_cancelados_parciais,database='silver'
)

In [77]:
format_type(df_ativos)
format_type(df_cancelamentos_integrais)
format_type(df_cancelamentos_parciais)

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,usuario_cancelamento,coverage_id,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_atualizacao,empresa
0,AYU0G87,9BSR6X200E3864540,4276,4276,<NA>,2494,13354,UNIDADE CURITIBA - INATIVA,AMILTON MARTUCCI,ATIVO,...,,83102,RASTREADOR,CANCELADO,2026-01-22,2025-01-28,2025-01-29,2025-01-29,2025-10-28,STCOOP
1,AYU0G87,9BSR6X200E3864540,4276,4276,<NA>,2494,13354,UNIDADE CURITIBA - INATIVA,AMILTON MARTUCCI,ATIVO,...,,83097,REPARAÇÃO A TERCEIROS,CANCELADO,2026-01-22,2025-01-28,2025-01-29,2025-01-29,2025-10-28,STCOOP
2,OAU3I79,93ZS3HUH0D8820730,23619,23619,<NA>,3623,13998,UNIDADE SINOP,QUEILA C. SANTOS,ATIVO,...,,87023,REPARAÇÃO A TERCEIROS,CANCELADO,2026-01-22,2025-03-28,2025-04-10,2025-04-10,2025-11-18,STCOOP
3,ARY6C41,9AA07102CAC088214,3347,0,3347,4293,15929,UNIDADE ITAPEJARA,DENILSO BRANCALIONE,ATIVO,...,,98678,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-01-22,2025-09-24,2025-10-09,2025-10-09,2025-12-08,STCOOP
4,ARY6C41,9AA07102CAC088214,3347,0,3347,4293,15929,UNIDADE ITAPEJARA,DENILSO BRANCALIONE,ATIVO,...,,98675,CASCO (R/SR),CANCELADO,2026-01-22,2025-09-24,2025-10-09,2025-10-09,2025-12-08,STCOOP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480,QJL8H90,97T0AN893KC005919,20368,0,20368,7771,13117,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,,100432,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-01-22,2025-03-27,2025-03-31,2025-03-31,2025-08-29,VIAVANTE
2481,ITO3645,9BSR6X200C3802775,16296163,16296163,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,,99737,VIDROS,CANCELADO,2026-01-22,2025-03-26,2025-03-31,2025-03-31,2025-06-23,VIAVANTE
2482,MLI3664,9BSR6X200D3836026,16296141,16296141,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,,99729,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-01-22,2025-03-26,2025-03-31,2025-03-31,2025-06-23,VIAVANTE
2483,CLH6I17,9BSR6X400B3690861,16341231,16341231,<NA>,7774,12964,YOU SAFER C,VICTORIA CHANG DE MORAIS,ATIVO,...,,99056,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-01-22,2025-03-26,2025-04-03,2025-04-03,2025-10-31,VIAVANTE


In [78]:

today_ts = pd.Timestamp.today().normalize()

if today_ts.weekday() == 0:  
    yesterday_ts = today_ts - dt.timedelta(days=3)
else:
    yesterday_ts = today_ts - dt.timedelta(days=1)
yesterday = yesterday_ts.strftime('%d-%m-%Y')


df_cancelamentos_integrais['data_cancelamento'] = pd.to_datetime(
	df_cancelamentos_integrais['data_cancelamento'], errors='coerce'
).dt.date

placas_canceladas_dia_anterior = df_cancelamentos_integrais.loc[
	df_cancelamentos_integrais['data_cancelamento'] == yesterday_ts.date(),
	'placa'
].unique().tolist()



In [79]:
df_cancelamentos_integrais['identificador'] = 'INTEGRAL'
df_cancelamentos_parciais['identificador'] = 'PARCIAL'

In [80]:
df_cancelamentos = pd.concat(
    [df_cancelamentos_integrais, df_cancelamentos_parciais], ignore_index=True
)

df_cancelamentos = df_cancelamentos.sort_values(
    by='data_cancelamento', ascending=False
).reset_index(drop=True)

df_cancelamentos.drop_duplicates(subset=['chassi', 'empresa', 'coverage_id'], keep='first')

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_cancelamento,empresa,identificador,data_atualizacao
0,RRA1D46,9AJG09400PAM59590,10145,0,10145,869,7554,UNIDADE MARINGÁ,JOSE LUIZ DOS SANTOS FILHO,FINALIZADO,...,CASCO (R/SR),ATIVO,2026-01-22,2024-12-27,2025-01-06,2025-01-06,2026-01-22,VIAVANTE,INTEGRAL,NaT
1,RRA1D52,9AJM06500PAM59591,10144,0,10144,869,7554,UNIDADE MARINGÁ,JOSE LUIZ DOS SANTOS FILHO,FINALIZADO,...,CASCO (R/SR),ATIVO,2026-01-22,2024-12-27,2025-01-06,2025-01-06,2026-01-22,VIAVANTE,INTEGRAL,NaT
2,BBF2B48,94BA0962GHV051634,12713,0,12713,5190,7758,MICRO B - JULIANO DE COSTA,JULIANO COSTA,FINALIZADO,...,RASTREADOR REBOQUE/SEMIRREBOQUE,ATIVO,2026-01-22,2025-01-06,2025-01-06,2025-01-06,2026-01-22,VIAVANTE,INTEGRAL,NaT
3,BBF0F69,94BA0712GHV051635,12712,0,12712,5190,7758,MICRO B - JULIANO DE COSTA,JULIANO COSTA,FINALIZADO,...,CASCO (R/SR),ATIVO,2026-01-22,2025-01-06,2025-01-06,2025-01-06,2026-01-22,VIAVANTE,INTEGRAL,NaT
4,FTT5I51,9BVRG20C8ME892747,11739,11739,<NA>,20774,12940,UNIDADE SÃO PAULO,MARCO KUABATA,FINALIZADO,...,RASTREADOR,ATIVO,2026-01-22,2024-12-27,2025-01-06,2025-01-06,2026-01-22,STCOOP,INTEGRAL,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37973,QJL8H90,97T0AN893KC005919,20368,0,20368,7771,13117,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-01-22,2025-03-27,2025-03-31,2025-03-31,NaN,VIAVANTE,PARCIAL,2025-08-29
37974,ITO3645,9BSR6X200C3802775,16296163,16296163,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,VIDROS,CANCELADO,2026-01-22,2025-03-26,2025-03-31,2025-03-31,NaN,VIAVANTE,PARCIAL,2025-06-23
37975,MLI3664,9BSR6X200D3836026,16296141,16296141,<NA>,7771,13040,UNIDADE JOINVILLE,AUGUSTINHO TADEU PIGNATEL,ATIVO,...,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-01-22,2025-03-26,2025-03-31,2025-03-31,NaN,VIAVANTE,PARCIAL,2025-06-23
37976,CLH6I17,9BSR6X400B3690861,16341231,16341231,<NA>,7774,12964,YOU SAFER C,VICTORIA CHANG DE MORAIS,ATIVO,...,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-01-22,2025-03-26,2025-04-03,2025-04-03,NaN,VIAVANTE,PARCIAL,2025-10-31


In [81]:
df_ativos = df_ativos.sort_values(
    by=['inicio_vig', 'data_ativacao'], ascending=False
).reset_index(drop=True)

In [82]:
today = dt.date.today()

yesterday = today - dt.timedelta(days=1)

In [83]:
if today.weekday() == 0:  # Segunda-feira
    # Pega desde sexta-feira até domingo
    sexta = today - dt.timedelta(days=3)
    domingo = today - dt.timedelta(days=1)
    ativos_mask = (df_ativos['inicio_vig'] >= sexta) & (df_ativos['inicio_vig'] <= domingo)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] >= sexta) & (df_cancelamentos['data_cancelamento'] <= domingo)
else:
    ativos_mask = (df_ativos['inicio_vig'] == yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] == yesterday)

ativados_seg = len(df_ativos[(df_ativos['empresa'] == 'SEGTRUCK') & ativos_mask])
ativados_st = len(df_ativos[(df_ativos['empresa'] == 'STCOOP') & ativos_mask])
ativados_viav = len(df_ativos[(df_ativos['empresa'] == 'VIAVANTE') & ativos_mask])
ativados_tag = len(df_ativos[(df_ativos['empresa'] == 'TAG') & ativos_mask])

cancelados_seg = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & cancelamentos_mask])
cancelados_st = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & cancelamentos_mask])
cancelados_viav = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & cancelamentos_mask])
cancelados_tag = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & cancelamentos_mask])

In [84]:
# FUNÇÃO PARA LIMPAR DADOS DA PLANILHA
def clear_sheet(sheet):
    max_row = sheet.max_row
    if max_row > 1:
        sheet.delete_rows(1,max_row)

In [85]:
template = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\template\TEMPLATE_BASE_ATIVACOES_CANCELAMENTOS.xlsx"

wb = openpyxl.load_workbook(template)

w1 = wb['ATIVOS']
w2 = wb['CANCELAMENTOS']
w3 = wb['BASE']
w4 = wb['RELATORIO']

clear_sheet(w1)
clear_sheet(w2)

In [86]:
# Fill NaN values: numeric columns with 0, string columns with 'N/A'
numeric_cols = df_ativos.select_dtypes(include=['number']).columns
string_cols = df_ativos.select_dtypes(include=['object', 'string']).columns

df_ativos[numeric_cols] = df_ativos[numeric_cols].fillna(0)
df_ativos[string_cols] = df_ativos[string_cols].fillna('N/A')

In [87]:
# Fill NaN values: numeric columns with 0, string columns with 'N/A'
numeric_cols = df_cancelamentos.select_dtypes(include=['number']).columns
string_cols = df_cancelamentos.select_dtypes(include=['object', 'string']).columns

df_cancelamentos[numeric_cols] = df_cancelamentos[numeric_cols].fillna(0)
df_cancelamentos[string_cols] = df_cancelamentos[string_cols].fillna('N/A')

In [88]:
# Inserir cabeçalhos
for c_idx, col_name in enumerate(df_ativos.columns, start=1):
    w1.cell(row=1, column=c_idx, value=col_name)

# Inserir dados (a partir da linha 2)
if not df_ativos.empty:
    for r_idx, row in enumerate(df_ativos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w1.cell(row=r_idx, column=c_idx, value=value)

In [89]:
# INSERIR CABEÇALHOS
for c_idx, col_name in enumerate(df_cancelamentos.columns, start=1):
    w2.cell(row=1, column=c_idx, value=col_name)

# ADICIONANDO OS DADOS NA ABA 'CANCELAMENTOS INTEGRAIS'
if  df_cancelamentos.empty == False:
    for r_idx, row in enumerate(df_cancelamentos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w2.cell(row=r_idx, column=c_idx, value=value)

### criando base

In [90]:
# ENCONTRANDO PRIMEIRA LINHA VAZIA NA COLUNA B ABA 'BASE' 


first_empty_row = 1
for row in range(1, w3.max_row + 1):
    if w3.cell(row=row, column=2).value is None:
        first_empty_row = row
        break
else:
    first_empty_row = w3.max_row + 1

# PEGAR DATA ATUAL NA PLANILHA
data_atual_planilha = w3['A' + str(first_empty_row)].value

# Certifique-se de que 'yesterday' e 'data_atual_planilha' sejam ambos date
if isinstance(data_atual_planilha, pd.Timestamp):
    data_atual_planilha = data_atual_planilha.date()
elif hasattr(data_atual_planilha, 'date'):
    data_atual_planilha = data_atual_planilha.date()
elif isinstance(data_atual_planilha, dt.datetime):
    data_atual_planilha = data_atual_planilha.date()
elif isinstance(data_atual_planilha, str):
    try:
        data_atual_planilha = pd.to_datetime(data_atual_planilha).date()
    except Exception:
        pass

# Também garanta que 'yesterday' é date
if isinstance(yesterday, pd.Timestamp):
    yesterday_date = yesterday.date()
elif isinstance(yesterday, dt.datetime):
    yesterday_date = yesterday.date()
else:
    yesterday_date = yesterday

# VERIFICA SE DATA NA PLANILHA = DATA DE ONTEM  
if data_atual_planilha == yesterday_date:
    # Filtra ativos até a data de ontem (inclusive)
    if "data_ativacao" in df_ativos.columns:
        # Supondo que data_ativacao já seja datetime ou string convertível
        data_ativacao_col = pd.to_datetime(df_ativos["data_ativacao"], errors='coerce')
        df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday_date)]
        qtd_ativos = df_ativos_filtrados['chassi'].nunique()
    else:
        qtd_ativos = df_ativos['chassi'].nunique()  # fallback: considera todos
    # Se hoje for segunda, preenche as duas linhas acima: DOMINGO e SÁBADO, depois insere qtd_ativos
    hoje = dt.date.today()
    if hoje.weekday() == 0 and first_empty_row >= 3:
        w3['B' + str(first_empty_row - 2)] = 'DOMINGO'
        w3['B' + str(first_empty_row - 1)] = 'SÁBADO'
        w3['B' + str(first_empty_row)] = qtd_ativos
        print('Registro de ativos preenchido para domingo, sábado e ativos na aba BASE!')
    else:
        w3['B' + str(first_empty_row)] = qtd_ativos
        print('Registro de ativos preenchido na aba BASE!')

Registro de ativos preenchido na aba BASE!


### criando resumo

In [91]:
# INSERINDO INFORMAÇÕES DE ATIVADOS E CANCELADOS

import datetime

# Garante que yesterday esteja normalizado
if isinstance(yesterday, pd.Timestamp):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.datetime):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.date):
    yest_date = yesterday
else:
    yest_date = pd.to_datetime(yesterday).date()

# Hoje
hoje = datetime.date.today()
dia_semana = hoje.weekday()  # 0 = segunda, ..., 6 = domingo

if dia_semana == 0:  # Segunda-feira
    # calcula sexta (3 dias atrás) e domingo (ontem)
    sexta = yest_date - datetime.timedelta(days=2)
    domingo = yest_date
    sexta_str = sexta.strftime('%d/%m/%Y')
    domingo_str = domingo.strftime('%d/%m/%Y')
    resumo_periodo = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
    w4['C2'] = resumo_periodo
else:
    w4['C2'] = yest_date.strftime('%d/%m/%Y')

w4['C3'] = qtd_ativos

w4['C6'] = ativados_seg
w4['C7'] = ativados_st
w4['C8'] = ativados_viav
w4['C9'] = ativados_tag

w4['D6'] = cancelados_seg
w4['D7'] = cancelados_st
w4['D8'] = cancelados_viav
w4['D9'] = cancelados_tag



### criando imagem do resumo

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Backend não-interativo para salvar imagens
import os

# --- 1. CONFIGURAÇÃO DOS DADOS (Usando valores das células w4) ---
# Totais calculados automaticamente
total_ativados = ativados_seg + ativados_st + ativados_viav + ativados_tag
total_cancelados = cancelados_seg + cancelados_st + cancelados_viav + cancelados_tag

# Variáveis do Cabeçalho Superior
if hasattr(yesterday, 'strftime'):
    if dia_semana == 0:  # Segunda-feira: mostra intervalo de sexta a domingo
        sexta = yest_date - datetime.timedelta(days=2)
        domingo = yest_date
        sexta_str = sexta.strftime('%d/%m/%Y')
        domingo_str = domingo.strftime('%d/%m/%Y')
        data_atual = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
    else:
        data_atual = yesterday.strftime('%d/%m/%Y')
else:
    try:
        y = str(yesterday)
        parts = y.split('-')
        if dia_semana == 0 and len(parts) == 3:
            # caso seja segunda e há info em formato yyyy-mm-dd
            sexta = yest_date - datetime.timedelta(days=2)
            domingo = yest_date
            sexta_str = sexta.strftime('%d/%m/%Y')
            domingo_str = domingo.strftime('%d/%m/%Y')
            data_atual = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
        elif len(parts) == 3:
            data_atual = f'{parts[2]}/{parts[1]}/{parts[0]}'
        else:
            data_atual = y
    except Exception:
        data_atual = str(yesterday)

# Total de ativos geral
try:
    if 'qtd_ativos' in locals() or 'qtd_ativos' in globals():
        total_ativos_geral = f"{qtd_ativos:,}".replace(',', '.')
    else:
        if "data_ativacao" in df_ativos.columns:
            data_ativacao_col = pd.to_datetime(df_ativos["data_ativacao"], errors='coerce')
            df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday_date)]
            total_ativos_geral = f"{df_ativos_filtrados['chassi'].nunique():,}".replace(',', '.')
        else:
            total_ativos_geral = f"{df_ativos['chassi'].nunique():,}".replace(',', '.')
except Exception:
    total_ativos_geral = f"{df_ativos['chassi'].nunique():,}".replace(',', '.')

# --- 2. ESTRUTURAÇÃO DAS TABELAS ---

# Cores
cor_fundo_cinza = '#E6E6E6'
cor_fundo_branco = '#FFFFFF'

# Dados da Tabela Principal (Inferior)
colunas_main = ['EMPRESA', 'ATIVADOS', 'CANCELADOS']
dados_main = [
    ['Segtruck', ativados_seg, cancelados_seg],
    ['Stcoop', ativados_st, cancelados_st],
    ['Viavante', ativados_viav, cancelados_viav],
    ['Tag', ativados_tag, cancelados_tag],
    ['Total', total_ativados, total_cancelados]
]

# Dados da Tabela de Cabeçalho (Superior)
dados_topo = [
    ['DATA', data_atual],
    ['TOTAL ATIVOS', total_ativos_geral]
]

# --- 3. CRIAÇÃO DO GRÁFICO ---
fig, ax = plt.subplots(figsize=(6, 5)) # Tamanho da figura
ax.axis('off')

# --- DESENHANDO A TABELA SUPERIOR (CABEÇALHO) ---
table_top = ax.table(
    cellText=dados_topo,
    cellLoc='center',
    loc='center',
    bbox=[0.0, 0.75, 1.0, 0.2]
)
# Estilização Cabeçalho: fundo cinza, texto negrito preto
for (row, col), cell in table_top.get_celld().items():
    cell.set_height(0.1)
    cell.set_text_props(fontweight='bold', color='#000000')  # negrito, preto
    cell.set_facecolor(cor_fundo_cinza if col == 0 else cor_fundo_branco)

# --- DESENHANDO A TABELA INFERIOR (PRINCIPAL) ---
table_main = ax.table(
    cellText=dados_main,
    colLabels=colunas_main,
    cellLoc='center',
    loc='center',
    bbox=[0.0, 0.0, 1.0, 0.7]
)

num_rows = len(dados_main)
num_cols = len(colunas_main)

for (row, col), cell in table_main.get_celld().items():
    cell.set_height(0.12)
    # Cabeçalho
    if row == 0:
        cell.set_facecolor(cor_fundo_cinza)
        cell.set_text_props(fontweight='bold', color='#000000')
    # Primeira coluna (nomes das empresas)
    elif col == 0:
        cell.set_facecolor(cor_fundo_cinza)
        cell.set_text_props(fontweight='bold', color='#000000')

    elif row == 3:
        cell.set_facecolor(cor_fundo_branco)
        cell.set_text_props(fontweight='normal', color='#000000')
    
    elif row == 5:
        cell.set_facecolor(cor_fundo_cinza)
        cell.set_text_props(fontweight='bold', color='#000000')
    # Restante: células em branco, fonte normal
    else:
        cell.set_facecolor(cor_fundo_branco)
        cell.set_text_props(fontweight='normal', color='#000000')

plt.tight_layout()

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img"
os.makedirs(output_path, exist_ok=True)
image_path = os.path.join(output_path, f'tabela_ativacoes_cancelamentos_{data_atual.replace('/', '-')}.png')
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Tabela salva em: {image_path}')

plt.close()

Tabela salva em: C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img\tabela_ativacoes_cancelamentos_21-01-2026.png


### construindo relação de chassis por unidade

In [93]:
df_ativos_ontem = df_ativos[df_ativos['inicio_vig'] == yesterday]

df_ativos_ontem_unidades = df_ativos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

df_ativos_ontem_unidades

df_cancelamentos_ontem = df_cancelamentos[df_cancelamentos['data_cancelamento'] == yesterday]

df_cancelamentos_ontem_unidades = df_cancelamentos_ontem.groupby('unidade')['chassi'].nunique().sort_values(ascending=False)

df_cancelamentos_ontem_unidades

unidade
UNIDADE LONDRINA                                                49
UNIDADE CUIABA                                                  26
UNIDADE MARINGÁ                                                 18
UNIDADE JOINVILLE                                               14
UNIDADE PONTA GROSSA                                            12
UNIDADE DOURADOS                                                11
UNIDADE CASCAVEL                                                10
UNIDADE RONDONOPOLIS                                             9
UNIDADE CAMPO GRANDE                                             7
UNIDADE MARINGÁ - VENDA ONLINE                                   7
UNIDADE VILHENA                                                  6
UNIDADE PORTO VELHO                                              5
MF - MICRO FRANQUEADO  J BATISTA VOLPATO REPRESENTACAO COMER     5
MICRO B - JULIANO DE COSTA                                       3
SR MARTINS                                            

### criando imagem das unidades

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Backend não-interativo para salvar imagens

# Formatação da data no padrão dd/mm/yyyy ou combinar sexta e domingo se segunda-feira
import datetime

# Obtém o dia da semana correspondente a 'yesterday'
# Se yesterday é datetime/date, use .weekday(). Caso contrário, tenta converter.
try:
    if hasattr(yesterday, 'weekday'):
        dia_semana = yesterday.weekday()
    else:
        dia_semana = datetime.datetime.strptime(str(yesterday), "%Y-%m-%d").weekday()
except Exception:
    dia_semana = None  # Não definido

if dia_semana == 0:  # Segunda-feira
    # calcula sexta (3 dias atrás) e domingo (ontem)
    if hasattr(yesterday, 'strftime'):
        domingo = yesterday
        sexta = yesterday - datetime.timedelta(days=2)
        sexta_str = sexta.strftime('%d/%m/%Y')
        domingo_str = domingo.strftime('%d/%m/%Y')
    else:
        y = str(yesterday)
        try:
            domingo = datetime.datetime.strptime(y, "%Y-%m-%d").date()
            sexta = domingo - datetime.timedelta(days=2)
            sexta_str = sexta.strftime('%d/%m/%Y')
            domingo_str = domingo.strftime('%d/%m/%Y')
        except Exception:
            # fallback apenas data como string
            sexta_str = y
            domingo_str = y
    yesterday_str = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
else:
    if hasattr(yesterday, 'strftime'):
        yesterday_str = yesterday.strftime('%d/%m/%Y')
    else:
        # Caso já seja string, tente rearranjar
        try:
            y = str(yesterday)
            # tenta yyyy-mm-dd
            parts = y.split('-')
            if len(parts) == 3:
                yesterday_str = f'{parts[2]}/{parts[1]}/{parts[0]}'
            else:
                yesterday_str = y
        except Exception:
            yesterday_str = str(yesterday)

# Criar figura com dois subplots lado a lado
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))  # Aumentei o tamanho horizontal da figura

# Gráfico 1: Ativos por Unidade
if not df_ativos_ontem_unidades.empty:
    df_ativos_ontem_unidades.plot(kind='barh', ax=ax1, color='#2ecc71', edgecolor='black', linewidth=0.5)
    ax1.set_title(f'Ativos por Unidade - {yesterday_str}', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax1.set_ylabel('', fontsize=11, fontweight='bold')  # Remove nome da legenda do eixo y
    # Remove as medições do eixo x
    ax1.set_xticks([])
    ax1.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
    ax1.invert_yaxis()  # Inverte para mostrar maior no topo

    # Aumenta o limite do eixo x para caber os data labels
    xmax = df_ativos_ontem_unidades.values.max() if len(df_ativos_ontem_unidades) > 0 else 1
    ax1.set_xlim(0, xmax + max(3, int(0.15 * xmax)))
    
    # Adicionar valores nas barras
    for i, v in enumerate(df_ativos_ontem_unidades.values):
        ax1.text(v + 0.7, i, str(int(v)), va='center', fontweight='bold')
else:
    ax1.text(0.5, 0.5, 'Sem dados disponíveis', ha='center', va='center', transform=ax1.transAxes, fontsize=12)
    ax1.set_title(f'Ativos por Unidade - {yesterday_str}', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax1.set_ylabel('', fontsize=11, fontweight='bold')  # Remove nome da legenda do eixo y
    ax1.set_xticks([])

# Gráfico 2: Cancelados por Unidade
if not df_cancelamentos_ontem_unidades.empty:
    df_cancelamentos_ontem_unidades.plot(kind='barh', ax=ax2, color='#e74c3c', edgecolor='black', linewidth=0.5)
    ax2.set_title(f'Cancelados por Unidade - {yesterday_str}', fontsize=14, fontweight='bold', pad=15)
    ax2.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax2.set_ylabel('', fontsize=11, fontweight='bold')  # Remove nome da legenda do eixo y
    # Remove as medições do eixo x
    ax2.set_xticks([])
    ax2.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
    ax2.invert_yaxis()  # Inverte para mostrar maior no topo

    # Aumenta o limite do eixo x para caber os data labels
    xmax2 = df_cancelamentos_ontem_unidades.values.max() if len(df_cancelamentos_ontem_unidades) > 0 else 1
    ax2.set_xlim(0, xmax2 + max(3, int(0.15 * xmax2)))
    
    # Adicionar valores nas barras
    for i, v in enumerate(df_cancelamentos_ontem_unidades.values):
        ax2.text(v + 0.7, i, str(int(v)), va='center', fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'Sem dados disponíveis', ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title(f'Cancelados por Unidade - {yesterday_str}', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Quantidade de Chassis', fontsize=11, fontweight='bold')
    ax2.set_ylabel('', fontsize=11, fontweight='bold')  # Remove nome da legenda do eixo y
    ax2.set_xticks([])

plt.tight_layout()

# Salvar a imagem
output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img"
os.makedirs(output_path, exist_ok=True)
image_path = os.path.join(output_path, f'graficos_unidades_{yesterday_str.replace("/", "-")}.png')
plt.savefig(image_path, dpi=300, bbox_inches='tight')
print(f'Gráficos salvos em: {image_path}')

plt.show()

C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_9524\942883000.py:33: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax1.grid(axis='x', alpha=0.3, linestyle='--', visible=False)
C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_9524\942883000.py:58: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax2.grid(axis='x', alpha=0.3, linestyle='--', visible=False)


Gráficos salvos em: C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img\graficos_unidades_21-01-2026.png


C:\Users\raphael.almeida\AppData\Local\Temp\ipykernel_9524\942883000.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### automação envio whatsapp

In [95]:
import time
import pyautogui

In [96]:
time.sleep(1)
pyautogui.hotkey('win', 'e')
time.sleep(4)
pyautogui.hotkey('ctrl', 'l') 
time.sleep(1.5)
pyautogui.typewrite(r'C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'f') 
time.sleep(2.5)
pyautogui.typewrite(r'graficos')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
pyautogui.press('down')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'c')
#whatsapp - primeiro arquivo
time.sleep(1.5)
pyautogui.press('win')
time.sleep(1.5)
pyautogui.typewrite('whatsapp')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(5)
pyautogui.hotkey('ctrl', 'f') 
time.sleep(1.5)
pyautogui.typewrite("raphael")
time.sleep(1.5)
pyautogui.press('down')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'v')
time.sleep(2.5)
pyautogui.press('enter')
time.sleep(1.5)
#whatsapp - segundo arquivo
pyautogui.hotkey('alt', 'tab')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'f') 
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'a')
time.sleep(2.5)
pyautogui.typewrite(r'tabela')
time.sleep(1.5)
pyautogui.press('enter')
time.sleep(1.5)
pyautogui.press('down')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'c')
time.sleep(1.5)
pyautogui.hotkey('alt', 'tab')
time.sleep(1.5)
pyautogui.hotkey('ctrl', 'v')
time.sleep(2.5)
pyautogui.press('enter')



In [97]:
import os

# save workbook to the full file path
wb.save(template)

name_file = fr'RELATORIO_ATIVACOES_CANCELAMENTOS_{yesterday}.xlsx'

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\reports"
os.makedirs(output_path, exist_ok=True)
output_full_path = os.path.join(output_path, name_file)

# save workbook to the full file path
shutil.copy(template, output_full_path)

name_file_2 = fr'RELATORIO_ATIVACOES_CANCELAMENTOS.xlsx'

path_sharepoint = r"C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel"
full_path_sharepoint = os.path.join(path_sharepoint, name_file_2)

# copy the saved file to the SharePoint folder (src, dst)
shutil.copy(output_full_path, full_path_sharepoint)

wb.close()